In [2]:
import os

# Trouve le dossier racine du projet en remontant depuis le dossier courant jusqu'à ce qu'on trouve 'data'
def find_project_root(current_path, target_folder="data"):
    path = current_path
    while True:
        if target_folder in os.listdir(path):
            return path
        new_path = os.path.dirname(path)
        if new_path == path:  # On est arrivé à la racine du disque sans trouver
            raise FileNotFoundError(f"Le dossier '{target_folder}' n'a pas été trouvé dans la hiérarchie des dossiers.")
        path = new_path

# Récupère le dossier courant
current_dir = os.getcwd()

# Trouve la racine du projet (dossier qui contient 'data')
project_root = find_project_root(current_dir, target_folder="data")

# Construit le chemin vers le dossier data/processed
data_processed_dir = os.path.join(project_root, "data", "processed")

# Chemin complet vers le fichier CSV
csv_path = os.path.join(data_processed_dir, "accidents_clean.csv")

print("Chemin absolu vers le fichier :", csv_path)

# Chargement du fichier
import pandas as pd
df = pd.read_csv(csv_path)
print(df.head())


Chemin absolu vers le fichier : /Users/alizeeblanchon/Documents/Data_Scientist/data_project/mai25_bds_accidents/data/processed/accidents_clean.csv
   obs  obsm  choc  manv  motor  place  catu  grav  sexe  trajet  ...  circ  \
0  0.0   2.0   5.0    23      1    2.0     2     2     2     0.0  ...   3.0   
1  0.0   2.0   5.0    23      1    1.0     1     2     2     5.0  ...   3.0   
2  1.0   0.0   3.0    11      1    1.0     1     1     1     0.0  ...   3.0   
3  4.0   0.0   1.0     0      1    1.0     1     2     2     0.0  ...   1.0   
4  0.0   2.0   1.0     2      1    1.0     1     1     1     0.0  ...   3.0   

   prof  plan surf  infra situ vma   age  equipements  catv_regroup  
0   1.0   2.0  1.0    2.0  1.0  70  17.0   [1.0, 0.0]           5.0  
1   1.0   2.0  1.0    2.0  1.0  70  26.0   [1.0, 0.0]           5.0  
2   1.0   2.0  1.0    2.0  1.0  70  60.0   [1.0, 0.0]           6.0  
3   4.0   2.0  1.0    0.0  1.0  70  25.0   [1.0, 0.0]           5.0  
4   1.0   3.0  1.0    0.0  1

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline

# === 1. Chargement des données

X = df.drop(columns=['grav'])
y = df['grav']

# === 2. Binarisation multilabel sur 'equipements'
mlb = MultiLabelBinarizer()
equip = pd.DataFrame(mlb.fit_transform(X['equipements']),
                     columns=[f'eq_{str(c)}' for c in mlb.classes_],
                     index=X.index)
X = pd.concat([X.drop(columns='equipements'), equip], axis=1)

# === 3. Préprocessing
equip_cols = list(equip.columns)
cols_to_exclude = ['dep'] + equip_cols
cols_to_ohe = [col for col in X.columns if col not in cols_to_exclude]

# Ajout d’un imputer pour robustesse
preprocessor = ColumnTransformer(transformers=[
    ('ohe', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cols_to_ohe),
    ('target_enc', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('target', TargetEncoder())
    ]), ['dep'])
], remainder='passthrough')

# === 4. Pipeline avec sous-échantillonnage
pipeline = ImbPipeline(steps=[
    ('encoding', preprocessor),
    ('scaling', StandardScaler()),  # utile pour logreg, même sur target encoded
    ('sampling', RandomUnderSampler(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
])

# === 5. Validation croisée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(pipeline, X, y, cv=cv)

# === 6. Évaluation
print("F1-score pondéré :", f1_score(y, y_pred, average='weighted'))
print("\nClassification Report :\n", classification_report(y, y_pred))

# === 7. Matrice de confusion
cm = confusion_matrix(y, y_pred)
classes = sorted(y.unique())
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Validation croisée avec sous-titre")
plt.show()

: 

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from category_encoders import TargetEncoder
from sklearn.impute import SimpleImputer
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import MultiLabelBinarizer
import seaborn as sns
import matplotlib.pyplot as plt

# Ajout de SmoteENN et GridSearchCV
# pour gérer le déséquilibre des classes et optimiser les hyperparamètres
# === 1. Chargement des données

X = df.drop(columns=['grav'])
y = df['grav']

# === 2. Binarisation multilabel sur 'equipements'
mlb = MultiLabelBinarizer()
equip = pd.DataFrame(mlb.fit_transform(X['equipements']),
                     columns=[f'eq_{str(c)}' for c in mlb.classes_],
                     index=X.index)
X = pd.concat([X.drop(columns='equipements'), equip], axis=1)

# === 3. Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === 4. Colonnes à encoder
equip_cols = list(equip.columns)
cols_to_exclude = ['dep'] + equip_cols
cols_to_ohe = [col for col in X.columns if col not in cols_to_exclude]

# === 5. Préprocesseur
preprocessor = ColumnTransformer(transformers=[
    ('ohe', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cols_to_ohe),
    ('target_enc', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('target', TargetEncoder())
    ]), ['dep'])
], remainder='passthrough')

# === 6. Pipeline avec SMOTEENN
pipeline = ImbPipeline(steps=[
    ('encoding', preprocessor),
    ('scaling', StandardScaler()),
    ('sampling', SMOTEENN(random_state=42)),
    ('classifier', LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs'))
])

# === 7. Grille de recherche d'hyperparamètres
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10],               # régularisation
    'classifier__penalty': ['l2'],                    # 'l1' non dispo avec lbfgs
    'classifier__solver': ['lbfgs'],                  # multinomial
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, scoring='f1_weighted', cv=cv, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# === 8. Évaluation sur test set
y_pred = grid_search.predict(X_test)

print("\n✅ Best parameters :", grid_search.best_params_)
print("F1-score pondéré :", f1_score(y_test, y_pred, average='weighted'))
print("\nClassification Report :\n", classification_report(y_test, y_pred))

# === 9. Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
classes = sorted(y.unique())
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title("Matrice de confusion - Test set (SMOTEENN + GridSearch)")
plt.tight_layout()
plt.show()


Fitting 5 folds for each of 4 candidates, totalling 20 fits
